<a href="https://colab.research.google.com/github/firstsignal/activation-geometry-sentiment/blob/main/universal_trajectory_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformer-lens


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.7 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=55b3cff2a1bee1fecc3e120c4359c64578555b362596b89a5a7fc38821c89dfb
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [12]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("pythia-70m")

# the matched pair from Chapter 2 — identical except one word
pair_pos = "The meal at the restaurant was absolutely wonderful and the evening felt complete."
pair_neg = "The meal at the restaurant was absolutely terrible and the evening felt complete."

tp = model.to_tokens(pair_pos); tn = model.to_tokens(pair_neg)
_, cp = model.run_with_cache(tp); _, cn = model.run_with_cache(tn)

# matched-pair sets for building the per-layer sentiment axis
pos_matched = [
    "The meal at the restaurant was absolutely wonderful.",
    "Her performance in the final act was brilliant.",
    "The weather on the coast stayed lovely all week.",
    "His speech at the ceremony sounded inspiring.",
    "The ending of the novel felt satisfying.",
]
neg_matched = [
    "The meal at the restaurant was absolutely terrible.",
    "Her performance in the final act was dreadful.",
    "The weather on the coast stayed miserable all week.",
    "His speech at the ceremony sounded tedious.",
    "The ending of the novel felt hollow.",
]

def resid_at(prompts, layer):
    vecs = []
    for p in prompts:
        _, c = model.run_with_cache(model.to_tokens(p))
        vecs.append(c["resid_post", layer][0].mean(dim=0))
    return torch.stack(vecs)

# sanity check the flip position before trusting [8:13]
toks = model.to_str_tokens(pair_pos)
for i, t in enumerate(toks): print(i, repr(t))


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer
0 'The'
1 ' meal'
2 ' at'
3 ' the'
4 ' restaurant'
5 ' was'
6 ' absolutely'
7 ' wonderful'
8 ' and'
9 ' the'
10 ' evening'
11 ' felt'
12 ' complete'
13 '.'


In [13]:
torch.manual_seed(3)
rand_axis = torch.randn(512); rand_axis = rand_axis / rand_axis.norm()

print(f"{'layer':>5} {'on-axis':>8} {'off-axis':>9} {'ratio':>7} {'on-random':>10}")
for L in range(6):
    dvec = cp["resid_post", L][0] - cn["resid_post", L][0]

    pa, na = resid_at(pos_matched, L), resid_at(neg_matched, L)
    axis = pa.mean(0) - na.mean(0); axis = axis / axis.norm()

    on_axis  = (dvec @ axis).abs()
    off_axis = (dvec - (dvec @ axis)[:, None] * axis[None, :]).norm(dim=-1)
    on_rand  = (dvec @ rand_axis).abs()

    s_on, s_off, s_rnd = on_axis[9:14].mean().item(), off_axis[9:14].mean().item(), on_rand[9:14].mean().item()
    print(f"{L:>5} {s_on:>8.3f} {s_off:>9.3f} {s_on/s_off:>7.3f} {s_rnd:>10.3f}")


layer  on-axis  off-axis   ratio  on-random
    0    0.068     0.443   0.153      0.023
    1    0.372     0.965   0.385      0.043
    2    0.788     2.050   0.384      0.110
    3    0.936     1.795   0.521      0.069
    4    1.512     2.593   0.583      0.069
    5    2.865     4.486   0.639      0.314


In [14]:
import torch

# ten FRESH matched pairs — not in the axis-building set
test_pairs = [
    ("The concert in the park sounded wonderful and the crowd stayed late.",
     "The concert in the park sounded terrible and the crowd stayed late."),
    ("Her garden looked beautiful after the rain stopped falling.",
     "Her garden looked dreadful after the rain stopped falling."),
    ("The service at the hotel was excellent and the staff seemed calm.",
     "The service at the hotel was awful and the staff seemed calm."),
    ("His first attempt at the recipe tasted delicious and the kitchen smelled good.",
     "His first attempt at the recipe tasted disgusting and the kitchen smelled good."),
    ("The view from the window seemed lovely in the morning light.",
     "The view from the window seemed miserable in the morning light."),
    ("The lecture on physics felt inspiring and the students asked questions.",
     "The lecture on physics felt tedious and the students asked questions."),
    ("The journey through the mountains was pleasant and the roads stayed clear.",
     "The journey through the mountains was horrible and the roads stayed clear."),
    ("The film about the ocean looked stunning and the music matched well.",
     "The film about the ocean looked boring and the music matched well."),
    ("The bread from the bakery smelled amazing and the queue moved quickly.",
     "The bread from the bakery smelled horrid and the queue moved quickly."),
    ("The report on the findings read brilliant and the figures looked clean.",
     "The report on the findings read hopeless and the figures looked clean."),
]

def flip_and_window(tp, tn):
    """Find the flipped position; return it and the downstream window. None if pair is unusable."""
    if tp.shape != tn.shape:
        return None, None                       # words tokenized to different lengths — skip
    diff = (tp[0] != tn[0]).nonzero().flatten()
    if len(diff) != 1:
        return None, None                       # more than one token differs — skip
    f = diff.item()
    return f, (f + 1, tp.shape[1])              # everything after the flip

results = {L: [] for L in range(6)}             # ratio per layer, per pair
rand_on = {L: [] for L in range(6)}

torch.manual_seed(3)
rand_axis = torch.randn(512); rand_axis = rand_axis / rand_axis.norm()

# axis per layer, built ONCE from the original matched sets (held out from test pairs)
axes = {}
for L in range(6):
    pa, na = resid_at(pos_matched, L), resid_at(neg_matched, L)
    a = pa.mean(0) - na.mean(0); axes[L] = a / a.norm()

used = 0
for s_pos, s_neg in test_pairs:
    tp, tn = model.to_tokens(s_pos), model.to_tokens(s_neg)
    f, win = flip_and_window(tp, tn)
    if f is None:
        print(f"SKIPPED (tokenization mismatch): {s_pos[:40]}...")
        continue
    used += 1
    _, cp2 = model.run_with_cache(tp); _, cn2 = model.run_with_cache(tn)
    for L in range(6):
        dvec = cp2["resid_post", L][0] - cn2["resid_post", L][0]
        axis = axes[L]
        on  = (dvec @ axis).abs()[win[0]:win[1]].mean().item()
        off = (dvec - (dvec @ axis)[:, None] * axis[None, :]).norm(dim=-1)[win[0]:win[1]].mean().item()
        rnd = (dvec @ rand_axis).abs()[win[0]:win[1]].mean().item()
        results[L].append(on / off)
        rand_on[L].append(rnd / off)

print(f"\npairs used: {used}/10")
print(f"{'layer':>5} {'ratio mean':>11} {'min':>7} {'max':>7} {'rand ratio':>11}")
for L in range(6):
    r = torch.tensor(results[L]); rr = torch.tensor(rand_on[L])
    print(f"{L:>5} {r.mean():>11.3f} {r.min():>7.3f} {r.max():>7.3f} {rr.mean():>11.3f}")


SKIPPED (tokenization mismatch): The bread from the bakery smelled amazin...

pairs used: 9/10
layer  ratio mean     min     max  rand ratio
    0       0.130   0.043   0.261       0.041
    1       0.242   0.158   0.349       0.036
    2       0.241   0.094   0.362       0.032
    3       0.341   0.222   0.490       0.032
    4       0.402   0.178   0.491       0.027
    5       0.379   0.140   0.484       0.038


flip-caused downstream change concentrates onto the sentiment direction as depth increases — from ~13% of the off-axis magnitude at layer 0 to ~40% by layers 3–5, an order of magnitude above a random direction — across 9 held-out matched pairs. The concentration saturates rather than compounding indefinitely, and varies substantially by sentence.